# sorted-computational-graph — faded example 1: Define get_parents to extract recipe parents

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sorted-computational-graph`. Running the beacon reports progress on the `Backprop: Sorted computation graph` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Sorted computation graph` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sorted-computational-graph`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sorted-computational-graph"
DD_SUBTOPIC = "Backprop: Sorted computation graph"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The topological sort needs a `get_children` function that, given a node, returns its parents (the nodes it depends on). For MiniTensor, this is: if `t.recipe is None` (a leaf), return []; otherwise return `list(t.recipe.parents.values())`. The `.values()` extracts the actual tensor objects; the dict keys are argument indices (0, 1, ...).

## Faded exercise 1

Complete `get_parents(node)` that returns the list of parent nodes from a MiniTensor's recipe.

1. If `node.recipe is None`, return an empty list.
2. Otherwise, return the parent tensors from `node.recipe.parents`.

The blank step is extracting the parent tensors from the recipe's parents dictionary.

**Fill in:** Return the parent tensors as a list by calling .values() on node.recipe.parents and converting to list.

In [ ]:
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None

def get_parents(node):
    if node.recipe is None:
        return []
    raise NotImplementedError()  # TODO: Return the parent tensors as a list by calling .values() on node.recipe.parents and converting to list.

a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [a, b])

print('get_parents(a):', [n.name for n in get_parents(a)])  # []
print('get_parents(b):', [n.name for n in get_parents(b)])  # ['a']
print('get_parents(c):', [n.name for n in get_parents(c)])  # ['a', 'b']


def _test():
    class FakeRecipe:
        def __init__(self, parents):
            self.parents = {i: p for i, p in enumerate(parents)}
    class FakeTensor:
        def __init__(self, name, parents=None):
            self.name = name
            self.recipe = FakeRecipe(parents) if parents else None

    a = FakeTensor('a')
    b = FakeTensor('b', [a])
    c = FakeTensor('c', [a, b])

    assert get_parents(a) == [], f'leaf should return [], got {get_parents(a)}'
    parents_b = get_parents(b)
    assert len(parents_b) == 1 and parents_b[0] is a
    parents_c = get_parents(c)
    assert len(parents_c) == 2
    assert parents_c[0] is a and parents_c[1] is b


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None

def get_parents(node):
    if node.recipe is None:
        return []
    return list(node.recipe.parents.values())

a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [a, b])

print('get_parents(a):', [n.name for n in get_parents(a)])
print('get_parents(b):', [n.name for n in get_parents(b)])
print('get_parents(c):', [n.name for n in get_parents(c)])
```
</details>